# 实验三：动态规划——仓库 AGV 路径规划 MDP

## 场景背景
在一个 4×4 的仓库网格中，AGV 需要从起点 (0,0) 到达目标 (3,3)。
网格中有两个障碍物，位于 (1,1) 和 (2,1)，AGV 不可进入。

AGV 可以执行四个动作：上、下、左、右。
由于地面打滑，执行动作时：
- 80% 概率按意图方向移动；
- 10% 概率向垂直方向偏移（左偏）；
- 10% 概率向垂直方向偏移（右偏）；
- 若移动方向撞墙或撞障碍，则停留在原地。

每走一步奖励 -1，到达目标奖励 +10。折扣因子 γ = 0.9。

## 实验目标
1. 理解马尔可夫决策过程（MDP）的要素：状态、动作、转移、奖励、折扣。
2. 掌握值迭代与策略迭代两种动态规划方法。
3. 比较两种方法的收敛过程与最终策略。

## 任务
1. 实现转移函数 `get_transition`。
2. 实现值迭代算法 `value_iteration`。
3. 实现策略评估 `policy_evaluation` 和策略改进 `policy_improvement`。
4. 实现策略迭代主循环 `policy_iteration`。
5. 输出最优价值函数矩阵和最优策略箭头图。
6. 比较两种方法的迭代次数与耗时。
7. 回答分析问题。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
%matplotlib inline

In [ ]:
# ==================== 网格与参数定义 ====================
ROWS, COLS = 4, 4
START = (0, 0)
GOAL = (3, 3)
OBSTACLES = {(1, 1), (2, 1)}

# 动作定义：0-上, 1-下, 2-左, 3-右
ACTIONS = [0, 1, 2, 3]
ACTION_NAMES = {0: '上', 1: '下', 2: '左', 3: '右'}

# 各动作的意图移动方向（行偏移, 列偏移）
MOVE = {
    0: (-1, 0),  # 上
    1: (1, 0),   # 下
    2: (0, -1),  # 左
    3: (0, 1)    # 右
}

# 垂直方向（用于打滑偏移）：每个动作对应的两个垂直动作
PERPENDICULAR = {
    0: [2, 3],  # 上 -> 左, 右
    1: [2, 3],  # 下 -> 左, 右
    2: [0, 1],  # 左 -> 上, 下
    3: [0, 1]   # 右 -> 上, 下
}

GAMMA = 0.9
STEP_REWARD = -1
GOAL_REWARD = 10

# 构建状态空间：所有非障碍格子
def build_states():
    states = []
    for r in range(ROWS):
        for c in range(COLS):
            if (r, c) not in OBSTACLES:
                states.append((r, c))
    return states

STATES = build_states()
print(f"状态数：{len(STATES)}")
print(f"状态列表：{STATES}")

In [ ]:
# ==================== 转移函数 ====================
def get_transition(state, action):
    """
    返回在 state 执行 action 后的转移列表：
    [(prob, next_state, reward), ...]

    规则：
    - 80% 按意图方向移动
    - 10% 向垂直方向偏移（左偏）
    - 10% 向垂直方向偏移（右偏）
    - 若移动后超出网格或进入障碍物，则停留在原地
    - 若 next_state 是目标，奖励为 GOAL_REWARD；否则为 STEP_REWARD
    """
    transitions = []
    r, c = state

    # TODO: 实现转移逻辑
    # 步骤提示：
    # 1. 获取意图方向 MOVE[action]，记为 (dr_int, dc_int)
    # 2. 获取垂直方向 PERPENDICULAR[action]，得到两个动作 a1, a2
    # 3. 构造三种移动：
    #    moves = [(0.8, MOVE[action]), (0.1, MOVE[a1]), (0.1, MOVE[a2])]
    # 4. 对每种移动 (prob, (dr, dc))：
    #    - 计算新位置 nr, nc = r + dr, c + dc
    #    - 若越界或 (nr, nc) 在 OBSTACLES 中，则 nr, nc = r, c（原地）
    #    - next_state = (nr, nc)
    #    - reward = GOAL_REWARD if next_state == GOAL else STEP_REWARD
    #    - 将 (prob, next_state, reward) 加入列表
    # 5. 合并相同 next_state 的概率（可选但推荐）

    return transitions

In [ ]:
# ==================== 值迭代 ====================
def value_iteration(theta=1e-8, max_iter=1000):
    """
    值迭代算法。
    返回：
        V       : 状态价值字典 {(r, c): value}
        policy  : 最优策略字典 {(r, c): action}
        n_iter  : 迭代次数
        elapsed : 耗时（秒）
    """
    V = {s: 0.0 for s in STATES}
    policy = {s: 0 for s in STATES}

    start_time = time.time()
    n_iter = 0

    for it in range(max_iter):
        delta = 0.0
        new_V = V.copy()

        for s in STATES:
            if s == GOAL:
                continue  # 目标状态价值固定为 0

            # TODO: 计算每个动作的期望价值 Q(s, a)
            # 对每个动作 a：
            #   q = Σ_{s'} P(s'|s,a) * [R(s,a,s') + γ * V(s')]
            # 其中 V(GOAL) = 0
            # 取最大的 q 作为 new_V[s]，并记录最优动作到 policy[s]
            # 更新 delta = max(delta, |new_V[s] - V[s]|)
            pass

        V = new_V
        n_iter = it + 1
        if delta < theta:
            break

    elapsed = time.time() - start_time
    return V, policy, n_iter, elapsed

In [ ]:
# ==================== 策略评估 ====================
def policy_evaluation(policy, theta=1e-8, max_iter=1000):
    """给定策略，迭代评估其状态价值"""
    V = {s: 0.0 for s in STATES}

    for it in range(max_iter):
        delta = 0.0
        new_V = V.copy()

        for s in STATES:
            if s == GOAL:
                continue

            # TODO: 根据策略 policy[s] 计算 V(s)
            # v = Σ_{s'} P(s'|s, π(s)) * [R(s,a,s') + γ * V(s')]
            # 更新 new_V[s]，并更新 delta
            pass

        V = new_V
        if delta < theta:
            break

    return V


# ==================== 策略改进 ====================
def policy_improvement(V):
    """根据 V 进行策略改进，返回新策略"""
    new_policy = {}

    for s in STATES:
        if s == GOAL:
            continue

        # TODO: 对每个动作 a 计算 Q(s, a)，选择 Q 最大的动作
        # new_policy[s] = argmax_a Q(s, a)
        pass

    return new_policy

In [ ]:
# ==================== 策略迭代 ====================
def policy_iteration(theta=1e-8, max_iter=1000):
    """
    策略迭代算法。
    返回：
        V       : 状态价值字典
        policy  : 最优策略字典
        n_iter  : 迭代次数
        elapsed : 耗时（秒）
    """
    # 初始化策略：所有非终止状态都选动作 0（上）
    policy = {s: 0 for s in STATES if s != GOAL}

    start_time = time.time()
    n_iter = 0

    for it in range(max_iter):
        # TODO: 实现策略迭代主循环
        # 1. 策略评估：V = policy_evaluation(policy)
        # 2. 策略改进：new_policy = policy_improvement(V)
        # 3. 检查策略是否稳定：所有状态的动作是否与旧策略相同
        # 4. 若稳定则跳出；否则 policy = new_policy
        pass

    elapsed = time.time() - start_time
    return V, policy, n_iter, elapsed

In [ ]:
# ==================== 运行与对比 ====================
print("=" * 50)
print("值迭代")
print("=" * 50)
V_vi, policy_vi, iter_vi, time_vi = value_iteration()
print(f"迭代次数：{iter_vi}")
print(f"耗时：{time_vi:.4f} 秒")

print("\n" + "=" * 50)
print("策略迭代")
print("=" * 50)
V_pi, policy_pi, iter_pi, time_pi = policy_iteration()
print(f"迭代次数：{iter_pi}")
print(f"耗时：{time_pi:.4f} 秒")

print("\n" + "=" * 50)
print("对比")
print("=" * 50)
print(f"值迭代  : 迭代 {iter_vi} 次，耗时 {time_vi:.4f} 秒")
print(f"策略迭代: 迭代 {iter_pi} 次，耗时 {time_pi:.4f} 秒")

In [ ]:
# ==================== 可视化 ====================
def plot_value_heatmap(V, title):
    """绘制价值函数热力图"""
    grid = np.full((ROWS, COLS), np.nan)
    for (r, c), v in V.items():
        grid[r, c] = v

    plt.figure(figsize=(6, 5))
    plt.imshow(grid, cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(label='V(s)')

    for r in range(ROWS):
        for c in range(COLS):
            if (r, c) in OBSTACLES:
                plt.text(c, r, '障碍', ha='center', va='center', fontsize=10)
            elif (r, c) == GOAL:
                plt.text(c, r, '终点', ha='center', va='center', fontsize=10)
            else:
                plt.text(c, r, f'{grid[r, c]:.2f}', ha='center', va='center', fontsize=8)

    plt.xticks(range(COLS))
    plt.yticks(range(ROWS))
    plt.title(title)
    plt.xlabel('列')
    plt.ylabel('行')
    plt.show()


def plot_policy(policy, title):
    """绘制策略箭头图"""
    arrows = {0: '↑', 1: '↓', 2: '←', 3: '→'}
    fig, ax = plt.subplots(figsize=(6, 5))

    for r in range(ROWS):
        for c in range(COLS):
            if (r, c) in OBSTACLES:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='gray'))
                ax.text(c, r, '障碍', ha='center', va='center', fontsize=10)
            elif (r, c) == GOAL:
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='lightgreen'))
                ax.text(c, r, '终点', ha='center', va='center', fontsize=10)
            else:
                ax.text(c, r, arrows[policy[(r, c)]], ha='center', va='center', fontsize=20)

    ax.set_xlim(-0.5, COLS - 0.5)
    ax.set_ylim(ROWS - 0.5, -0.5)
    ax.set_xticks(range(COLS))
    ax.set_yticks(range(ROWS))
    ax.set_title(title)
    ax.set_xlabel('列')
    ax.set_ylabel('行')
    ax.grid(True)
    plt.show()


if V_vi:
    plot_value_heatmap(V_vi, '值迭代：最优价值函数 V(s)')
    plot_policy(policy_vi, '值迭代：最优策略')

## 分析问题

1. 障碍物附近的状态，最优策略有什么特点？为什么？
2. 折扣因子变为 0.5 或 0.99 时，最优策略会如何变化？请修改 `GAMMA` 后重新运行实验。
3. 值迭代和策略迭代哪种方法迭代次数更少？哪种方法总耗时更少？为什么？
4. 如果将打滑概率改为 0（完全确定），最优策略会怎样变化？

请在此处写下你的分析。